# Topic 4: Caching & Persistence

> **Phase 3 → Week 5 → Topic 4**

---

## Why Caching Exists — The Recomputation Problem

Spark DataFrames/RDDs are **lazy**. Every time you call an action, Spark recomputes from scratch:

```
raw_data = spark.read.parquet("huge_file")  # 100GB
cleaned  = raw_data.filter(...).withColumn(...).join(...)   # expensive transform

# Without cache:
cleaned.count()         # reads 100GB + computes transform
cleaned.show()          # reads 100GB + computes transform AGAIN
cleaned.write.parquet() # reads 100GB + computes transform AGAIN
                        # = 3x the work!

# With cache:
cleaned.cache()         # marks for caching
cleaned.count()         # reads 100GB, transforms, STORES IN MEMORY
cleaned.show()          # reads from memory cache (fast)
cleaned.write.parquet() # reads from memory cache (fast)
                        # = 1x read + 2x cache hits
```

---

## cache() vs persist()

```
df.cache()   ≡   df.persist(StorageLevel.MEMORY_AND_DISK)

cache() is a shortcut for persist() with the default storage level.
Use persist() when you need a different storage level.
```

## Storage Levels

| Storage Level | Where | Serialized? | Replicated? | When to Use |
|--------------|-------|-------------|-------------|-------------|
| `MEMORY_ONLY` | RAM only | No | No | Fast, data fits in RAM |
| `MEMORY_AND_DISK` | RAM → spill to disk | No | No | Default. Good for most cases |
| `MEMORY_ONLY_SER` | RAM, serialized | Yes | No | Less RAM but slower (deserialization cost) |
| `MEMORY_AND_DISK_SER` | RAM + disk, serialized | Yes | No | Balance of RAM savings and speed |
| `DISK_ONLY` | Disk only | Yes | No | Data too large for RAM |
| `MEMORY_AND_DISK_2` | RAM + disk | No | Yes (2 copies) | Fault tolerance needed |
| `OFF_HEAP` | Off-heap memory | Yes | No | Avoid JVM GC pressure |

---

## When to Cache (and When NOT to)

**Cache when:**
- DataFrame used multiple times (>2 actions on same DF)
- Expensive computation (big joins, complex aggregations, ML feature transforms)
- Iterative algorithms (ML training loops)

**Do NOT cache when:**
- DataFrame used only once — wasted memory
- DataFrame is read directly from fast storage (SSD Parquet) — I/O already fast
- Data doesn't fit in memory — spilling to disk may be slower than just recomputing
- Simple transforms — recomputing may be cheaper than memory pressure from caching

---

## Interview Cheat Sheet

**Q: What's the difference between cache() and persist()?**
> `cache()` is a shorthand for `persist(StorageLevel.MEMORY_AND_DISK)`. Use `persist()` when you need a specific storage level — for example, `MEMORY_ONLY` for maximum speed, or `DISK_ONLY` when RAM is scarce.

**Q: Is cache() immediate? When does data actually get cached?**
> No. `cache()` is lazy — it just marks the DataFrame for caching. Data is only materialized and stored when the first **action** is called on that DataFrame (or a downstream DataFrame derived from it). You need to trigger an action like `.count()` to actually populate the cache.

**Q: What happens if cached data doesn't fit in memory?**
> With `MEMORY_AND_DISK` (the default), partitions that don't fit in RAM are spilled to disk. On cache hit, Spark reads from disk instead of memory — slower than memory but faster than recomputing from scratch (especially if the original computation involves network I/O). With `MEMORY_ONLY`, partitions that don't fit are dropped and recomputed on access.

**Q: How do you unpersist a cached DataFrame?**
> Call `df.unpersist()`. This releases the memory. Always unpersist DataFrames you no longer need — otherwise they occupy memory for the entire session and can cause OOM errors or GC pressure. `unpersist(blocking=True)` waits for the memory to be fully freed.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.storagelevel import StorageLevel
import time

spark = SparkSession.builder \
    .appName("Week5 - Caching") \
    .master("local[4]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

customers = spark.read.csv("data/customers.csv", header=True, inferSchema=True)
orders    = spark.read.csv("data/orders.csv",    header=True, inferSchema=True)
products  = spark.read.csv("data/products.csv",  header=True, inferSchema=True)
print("Ready")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/13 15:58:05 WARN Utils: Your hostname, kali, resolves to a loopback address: 127.0.1.1; using 10.237.41.61 instead (on interface wlan0)
26/06/13 15:58:05 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/06/13 15:58:06 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Ready


## Part 1: cache() and persist() Basics

In [2]:
# Create an expensive transformation
enriched = (
    orders
    .join(customers, on="customer_id", how="left")
    .join(products,  on="product_id",  how="left")
    .withColumn("revenue_category",
        F.when(F.col("amount") >= 1000, "High")
         .when(F.col("amount") >= 200, "Medium")
         .otherwise("Low"))
)

# WITHOUT cache — recomputed 3 times
print("=== WITHOUT CACHE ===")
t0 = time.time()
count1 = enriched.count()
time1 = time.time() - t0

t0 = time.time()
count2 = enriched.filter(F.col("revenue_category") == "High").count()
time2 = time.time() - t0

t0 = time.time()
summary = enriched.groupBy("category").agg(F.sum("amount")).collect()
time3 = time.time() - t0

print(f"  Action 1: {time1:.3f}s")
print(f"  Action 2: {time2:.3f}s")
print(f"  Action 3: {time3:.3f}s")
print(f"  Total: {time1+time2+time3:.3f}s")

=== WITHOUT CACHE ===


  Action 1: 1.412s
  Action 2: 1.001s
  Action 3: 1.233s
  Total: 3.645s


In [3]:
# WITH cache — computed once, served from memory
print("=== WITH CACHE ===")
enriched.cache()       # marks for caching (lazy!)

t0 = time.time()
count1 = enriched.count()   # first action — populates cache
cache_fill_time = time.time() - t0

t0 = time.time()
count2 = enriched.filter(F.col("revenue_category") == "High").count()  # from cache
cache_hit_time2 = time.time() - t0

t0 = time.time()
summary = enriched.groupBy("category").agg(F.sum("amount")).collect()  # from cache
cache_hit_time3 = time.time() - t0

print(f"  Action 1 (fills cache): {cache_fill_time:.3f}s")
print(f"  Action 2 (cache hit):   {cache_hit_time2:.3f}s")
print(f"  Action 3 (cache hit):   {cache_hit_time3:.3f}s")
print(f"  Total: {cache_fill_time+cache_hit_time2+cache_hit_time3:.3f}s")

enriched.unpersist()  # always clean up!

=== WITH CACHE ===


  Action 1 (fills cache): 1.023s
  Action 2 (cache hit):   0.337s
  Action 3 (cache hit):   0.563s
  Total: 1.923s


DataFrame[product_id: string, customer_id: int, order_id: string, quantity: int, order_date: date, status: string, amount: double, name: string, city: string, country: string, signup_date: date, tier: string, name: string, category: string, price: double, stock: int, supplier_id: string, revenue_category: string]

In [4]:
# Check if a DataFrame is cached
df_test = orders.filter(F.col("amount") > 100)

print(f"Is cached before cache(): {df_test.is_cached}")

df_test.cache()
df_test.count()  # trigger materialization
print(f"Is cached after cache():  {df_test.is_cached}")

df_test.unpersist()
print(f"Is cached after unpersist(): {df_test.is_cached}")

Is cached before cache(): False


Is cached after cache():  True
Is cached after unpersist(): False


## Part 2: Storage Levels

In [5]:
# Available storage levels in PySpark
print("Storage Level Reference:")
levels = [
    ("MEMORY_ONLY",           StorageLevel.MEMORY_ONLY),
    ("MEMORY_AND_DISK",       StorageLevel.MEMORY_AND_DISK),
    ("DISK_ONLY",             StorageLevel.DISK_ONLY),
    ("MEMORY_AND_DISK_2",     StorageLevel.MEMORY_AND_DISK_2),
    ("OFF_HEAP",              StorageLevel.OFF_HEAP),
]

print(f"{'Level':<26} {'useMemory':>10} {'useDisk':>8} {'replication':>12}")
print("-" * 60)
for name, level in levels:
    print(f"{name:<26} {str(level.useMemory):>10} {str(level.useDisk):>8} {level.replication:>12}")


Storage Level Reference:
Level                       useMemory  useDisk  replication
------------------------------------------------------------
MEMORY_ONLY                      True    False            1
MEMORY_AND_DISK                  True     True            1
DISK_ONLY                       False     True            1
MEMORY_AND_DISK_2                True     True            2
OFF_HEAP                         True     True            1


In [6]:
# Using persist() with explicit storage level
df = enriched

# For data that might not fit in RAM, use MEMORY_AND_DISK (same as cache())
df.persist(StorageLevel.MEMORY_AND_DISK)
df.count()  # materialize
print(f"Persisted with MEMORY_AND_DISK, is_cached: {df.is_cached}")
df.unpersist()

# For ML iterative algorithms where speed > memory: MEMORY_ONLY
df.persist(StorageLevel.MEMORY_ONLY)
df.count()
print(f"Persisted with MEMORY_ONLY, is_cached: {df.is_cached}")
df.unpersist()

# For large data with limited RAM: DISK_ONLY
df.persist(StorageLevel.DISK_ONLY)
df.count()
print(f"Persisted with DISK_ONLY, is_cached: {df.is_cached}")
df.unpersist()

Persisted with MEMORY_AND_DISK, is_cached: True


Persisted with MEMORY_ONLY, is_cached: True


Persisted with DISK_ONLY, is_cached: True


DataFrame[product_id: string, customer_id: int, order_id: string, quantity: int, order_date: date, status: string, amount: double, name: string, city: string, country: string, signup_date: date, tier: string, name: string, category: string, price: double, stock: int, supplier_id: string, revenue_category: string]

## Part 3: Caching Best Practices

In [7]:
# PATTERN 1: Cache a shared base DataFrame in a pipeline
# Scenario: multiple downstream analyses from one expensive join

base_df = (
    orders
    .join(customers, on="customer_id", how="left")
    .join(products,  on="product_id",  how="left")
)
base_df.cache()
base_df.count()  # materialize

# Multiple analyses — all served from cache
analysis1 = base_df.groupBy("category").agg(F.sum("amount").alias("revenue"))
analysis2 = base_df.groupBy("country").agg(F.count("order_id").alias("orders"))
analysis3 = base_df.filter(F.col("status") == "delivered") \
                    .groupBy("tier").agg(F.avg("amount").alias("avg_spend"))

analysis1.show()
analysis2.show()
analysis3.show()

# Clean up when done
base_df.unpersist()
print("Cache released.")

+-----------+------------------+
|   category|           revenue|
+-----------+------------------+
|      Books|359.90999999999997|
|Electronics| 7349.799999999997|
|  Furniture|           1469.94|
+-----------+------------------+



+-----------+------+
|    country|orders|
+-----------+------+
|South Korea|     1|
|         UK|     2|
|      Japan|     2|
|      Italy|     1|
|        USA|     3|
|    Germany|     2|
|       NULL|     1|
|      India|     5|
|      China|     2|
|    Nigeria|     1|
+-----------+------+



+------+---------+
|  tier|avg_spend|
+------+---------+
|  Gold|831.23375|
|Silver|  120.976|
|Bronze|    59.98|
|  NULL|  1299.99|
+------+---------+

Cache released.


In [8]:
# PATTERN 2: Don't cache DataFrames used only once
# BAD: wastes memory
orders_cached = orders.cache()
result = orders_cached.filter(F.col("amount") > 100).show()  # only used once
orders_cached.unpersist()

# GOOD: just use it directly
orders.filter(F.col("amount") > 100).show()

+--------+-----------+----------+--------+----------+----------+-------+
|order_id|customer_id|product_id|quantity|order_date|    status| amount|
+--------+-----------+----------+--------+----------+----------+-------+
|    O001|          1|      P001|       1|2023-06-01| delivered|1299.99|
|    O004|          3|      P001|       1|2023-06-12| delivered|1299.99|
|    O005|          3|      P004|       2|2023-06-12| delivered| 599.98|
|    O007|          5|      P008|       1|2023-06-18| delivered| 199.99|
|    O008|          6|      P003|       1|2023-06-20| delivered| 449.99|
|    O010|          8|      P002|       5|2023-06-25| delivered| 149.95|
|    O012|         10|      P001|       2|2023-07-01| delivered|2599.98|
|    O013|          1|      P008|       1|2023-07-05| delivered| 199.99|
|    O015|          6|      P005|       3|2023-07-10| delivered| 119.97|
|    O016|         11|      P001|       1|2023-07-12| delivered|1299.99|
|    O017|          2|      P009|       2|2023-07-1

+--------+-----------+----------+--------+----------+----------+-------+
|order_id|customer_id|product_id|quantity|order_date|    status| amount|
+--------+-----------+----------+--------+----------+----------+-------+
|    O001|          1|      P001|       1|2023-06-01| delivered|1299.99|
|    O004|          3|      P001|       1|2023-06-12| delivered|1299.99|
|    O005|          3|      P004|       2|2023-06-12| delivered| 599.98|
|    O007|          5|      P008|       1|2023-06-18| delivered| 199.99|
|    O008|          6|      P003|       1|2023-06-20| delivered| 449.99|
|    O010|          8|      P002|       5|2023-06-25| delivered| 149.95|
|    O012|         10|      P001|       2|2023-07-01| delivered|2599.98|
|    O013|          1|      P008|       1|2023-07-05| delivered| 199.99|
|    O015|          6|      P005|       3|2023-07-10| delivered| 119.97|
|    O016|         11|      P001|       1|2023-07-12| delivered|1299.99|
|    O017|          2|      P009|       2|2023-07-1

In [9]:
# PATTERN 3: Checkpoint — break lineage for iterative algorithms
# cache() stores in memory but KEEPS the lineage (full computation graph)
# checkpoint() writes to disk AND TRUNCATES the lineage — safer for deep pipelines

import os
spark.sparkContext.setCheckpointDir("/tmp/spark_checkpoint")

# Simulate iterative pipeline
df = orders
for i in range(3):  # each iteration adds to the lineage
    df = df.withColumn(f"iter_{i}", F.col("amount") + i)

print("Lineage BEFORE checkpoint (3 withColumn transforms):")
df.explain()

# Checkpoint breaks the lineage
df_checkpointed = df.checkpoint()  # writes to /tmp/spark_checkpoint

print("\nPlan AFTER checkpoint (lineage cut — starts fresh from disk):")
df_checkpointed.explain()

Lineage BEFORE checkpoint (3 withColumn transforms):


== Physical Plan ==
*(1) Project [order_id#40, customer_id#41, product_id#42, quantity#43, order_date#44, status#45, amount#46, (amount#46 + 0.0) AS iter_0#5813, (amount#46 + 1.0) AS iter_1#5814, (amount#46 + 2.0) AS iter_2#5815]
+- FileScan csv [order_id#40,customer_id#41,product_id#42,quantity#43,order_date#44,status#45,amount#46] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/home/khushal/Desktop/learn_spark/week05/data/orders.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<order_id:string,customer_id:int,product_id:string,quantity:int,order_date:date,status:stri...





Plan AFTER checkpoint (lineage cut — starts fresh from disk):
== Physical Plan ==
*(1) Scan ExistingRDD[order_id#40,customer_id#41,product_id#42,quantity#43,order_date#44,status#45,amount#46,iter_0#5813,iter_1#5814,iter_2#5815]




## Part 4: Monitoring Cache Usage in Spark UI

In [10]:
# Cache a named DataFrame and check storage tab in UI
big_df = spark.range(1_000_000).withColumn("val", F.rand() * 1000)
big_df.cache()
big_df.count()  # materialize

print("""
Now go to Spark UI → http://localhost:4040 → Storage tab
You should see:
  - RDD Name (the DataFrame's internal RDD name)
  - Storage Level (MEMORY_AND_DISK)
  - Fraction Cached (% of partitions in memory)
  - Size in Memory
  - Size on Disk (if spilled)
""")

big_df.unpersist(blocking=True)  # blocking=True waits for memory to free
print("Cache freed. Check Storage tab — entry should be gone.")


Now go to Spark UI → http://localhost:4040 → Storage tab
You should see:
  - RDD Name (the DataFrame's internal RDD name)
  - Storage Level (MEMORY_AND_DISK)
  - Fraction Cached (% of partitions in memory)
  - Size in Memory
  - Size on Disk (if spilled)

Cache freed. Check Storage tab — entry should be gone.


## Exercises

1. Create a pipeline that: reads orders, joins with products, computes category-level stats. Cache the joined DataFrame and run 3 different aggregations on it. Time the difference between cached and uncached.
2. What's the difference between `df.cache()` and `df.persist(StorageLevel.MEMORY_ONLY)`? When would you choose `MEMORY_ONLY` over the default?
3. Write code that simulates an iterative algorithm (5 iterations of adding a column), then uses checkpoint to break the lineage. Compare the explain() output before and after.
4. Explain why this code is wasteful:
   ```python
   df.cache()
   print(df.count())
   df.unpersist()
   ```
5. When should you use `df.checkpoint()` instead of `df.cache()`?

In [11]:
# Exercise 1: Cache comparison timing
def run_pipeline(use_cache: bool):
    joined = orders.join(products, on="product_id", how="left") \
                   .join(customers, on="customer_id", how="left")
    if use_cache:
        joined.cache()
        joined.count()  # materialize
    
    t0 = time.time()
    r1 = joined.groupBy("category").agg(F.sum("amount")).count()
    r2 = joined.groupBy("country").agg(F.count("order_id")).count()
    r3 = joined.groupBy("status").agg(F.avg("amount")).count()
    elapsed = time.time() - t0
    
    if use_cache:
        joined.unpersist()
    return elapsed

import time
no_cache_time = run_pipeline(use_cache=False)
cache_time    = run_pipeline(use_cache=True)
print(f"Without cache: {no_cache_time:.3f}s")
print(f"With cache:    {cache_time:.3f}s")

Without cache: 1.260s
With cache:    0.774s
